# MEGA → Google Drive Transfer via Colab

Run cells in order: **Cell 1 → Cell 2 → Cell 3 → Cell 4**.
You must keep the Colab browser tab open during the transfer.

In [ ]:
# ===== Cell 1: Install rclone =====
%%bash
curl -s https://rclone.org/install.sh | sudo bash
rclone --version

In [ ]:
# ===== Cell 2: Helper — Obscure your MEGA password =====
# Run this ONLY to get the obscured password, then paste it into Cell 3.
%%bash
# Example: rclone obscure "your-actual-mega-password"
# rclone obscure "PUT_YOUR_MEGA_PASSWORD_HERE"

In [ ]:
# ===== Cell 3: Configure rclone remotes =====
# Step A — Fill in your values below:
MEGA_USER = "YOUR_MEGA_EMAIL"
MEGA_PASS_OBSCURED = "PASTE_OUTPUT_FROM_CELL_2_HERE"
GDRIVE_TOKEN = """\
PASTE_YOUR_GDRIVE_OAUTH_TOKEN_HERE
Get it by running on your Mac:
  rclone authorize \"drive\"
A browser will open. After authorizing, paste the full JSON output here.
"""

# Step B — Write the config file
import os

RCLONE_CONF = f"""[mega]
type = mega
user = {MEGA_USER}
pass = {MEGA_PASS_OBSCURED}

[gdrive]
type = drive
scope = drive
token = {GDRIVE_TOKEN}
"""

os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write(RCLONE_CONF.strip() + "\n")

print("✅ Config written.")

# Step C — Verify remotes
import subprocess
result = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
print(f"Remotes: {result.stdout.strip()}")

result = subprocess.run(["rclone", "lsd", "mega:"], capture_output=True, text=True, timeout=30)
print(f"MEGA root listing ({result.returncode}): {result.stdout.strip() or result.stderr.strip()}")

result = subprocess.run(["rclone", "lsd", "gdrive:"], capture_output=True, text=True, timeout=30)
print(f"GDrive root listing ({result.returncode}): {result.stdout.strip() or result.stderr.strip()}")


In [ ]:
# ===== Cell 4: Dry run first (safe, no data transferred) =====
import subprocess

# Update source/dest paths if needed
SRC = "mega:Cloud/PMVDL/"
DST = "gdrive:PMVDL/"

cmd = [
    "rclone", "copy", SRC, DST,
    "--dry-run", "-P",
    "--transfers", "4",
    "--tpslimit", "10",
    "--retries", "3",
    "--drive-chunk-size", "64M",
]
print(f"Running: {' '.join(cmd)}\n")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# ===== Cell 5: Actual transfer =====
# This cell runs the real copy. Keep the browser tab open.
import subprocess
import os

SRC = "mega:Cloud/PMVDL/"
DST = "gdrive:PMVDL/"

cmd = [
    "rclone", "copy", SRC, DST,
    "-P",  # --progress (live display)
    "--transfers", "4",
    "--tpslimit", "10",
    "--retries", "3",
    "--retries-sleep", "5s",
    "--drive-chunk-size", "64M",
    "--drive-stop-on-upload-limit",
]

print(f"Starting transfer: {SRC} → {DST}\n")
os.execvp("rclone", cmd)  # replaces this process so output streams live